# Consolidation + Per-Class LoRA Compression for Qwen1.5-0.5B

**Research prototype.** Frozen Qwen1.5-0.5B teacher → consolidated student (shared per-class backbone + per-layer rank-r LoRA) distilled to match it.

> ⚠️ **Set the GPU first:** Runtime → Change runtime type → **T4 GPU**, then Run all.

Reading the result: the question is *not* whether the student beats the teacher (it won't). It's whether a consolidated point sits **left of / below** the 4-bit nf4 baseline in `frontier.png` (lower perplexity = better).

In [ ]:
# 1. Clone the repo
!git clone https://github.com/sinha-k-prat/consolidated-qwen.git
%cd consolidated-qwen

In [ ]:
# 2. Install pinned dependencies (Colab-tested versions)
!pip install -q -r requirements.txt

In [ ]:
# 3. Confirm we actually have a GPU (expect a Tesla T4). If this errors or shows
#    no GPU, go to Runtime > Change runtime type > T4 GPU and re-run.
!nvidia-smi

In [ ]:
# 4. (Optional but recommended) Smoke-test the whole pipeline in ~1 min:
#    10 steps/rank on tiny data, catches shape bugs before the real run.
!python run_sweep.py --smoke-test --ranks 8

In [ ]:
# 5. Real proof-of-concept sweep over ranks [4, 8, 16, 32].
#    Small step counts keep this under ~2h on a T4. --fp16 saves memory.
#    Tweak --steps / --ranks to trade runtime for quality.
!python run_sweep.py --fp16 --ranks 4 8 16 32 --steps 300

In [ ]:
# 6. Show the frontier inline (perplexity vs storage size).
from IPython.display import Image, display
import json
with open('results.json') as f:
    print(json.dumps(json.load(f), indent=2))
display(Image('frontier.png'))

In [ ]:
# 7. Held-out CE: standard Qwen (teacher) vs consolidated student over 100
#    held-out sequences. Verifies the per-class weights are LITERALLY shared.
!python compare_ce.py --checkpoint checkpoints/student_rank8.pt --num-seqs 100
import json
with open('ce_compare.json') as f:
    r = json.load(f)
print('shared-backbone verified:', r['shared_backbone_verified'])
print('teacher mean CE     :', round(r['teacher']['mean_ce'], 4))
print('consolidated mean CE:', round(r['consolidated']['mean_ce'], 4))
print('mean CE gap (s-t)   :', round(r['mean_ce_gap_student_minus_teacher'], 4))

In [ ]:
# 8. Per-layer / per-class divergence from the shared mean (= the adapter
#    delta scale*A@B). Bright cells = layers/classes that resist sharing.
!python divergence.py --checkpoint checkpoints/student_rank8.pt
from IPython.display import Image, display
display(Image('divergence.png'))

## Save & reuse weights on the Hugging Face Hub

Colab's disk is **ephemeral** — checkpoints vanish when the runtime recycles. Push the
small consolidated checkpoint (~33 MB; just backbones + adapters + biases) to a HF model
repo so you can pull it into any later session and run inference. Set `HF_USER` to your
Hugging Face username in the next cell.

In [ ]:
# 9. Log in to Hugging Face and push the weights.
#    IMPORTANT: paste a token with the "Write" role (huggingface.co/settings/tokens
#    -> New token -> Write). A read-only token gives a 401 on repo creation.
from huggingface_hub import notebook_login, whoami
notebook_login()

HF_USER = whoami()["name"]          # auto-detected from your token (no typo risk)
REPO_ID = f"{HF_USER}/consolidated-qwen-rank8"
print("pushing to:", REPO_ID)
!python hub.py push --checkpoint checkpoints/student_rank8.pt --repo-id {REPO_ID}

In [ ]:
# 10. Pull the weights back from the Hub and play with inference.
#     (Works in any later session — re-run cells 1-2 first, then this.)
from hub import pull
from infer import generate

model, tok = pull(REPO_ID)        # re-downloads base Qwen + your consolidated tensors
print(generate(model, tok,
               "The key idea behind weight consolidation is",
               max_new_tokens=80))